# Lumen Detection Demo

This notebook demonstrates a simple computer vision algorithm to detect the colon lumen in a colonoscopy video.

## 1. Setup

First, we need to install the necessary Python libraries. We only need OpenCV for video processing and NumPy for numerical operations.

In [ ]:
!pip install -q opencv-python-headless numpy

## 2. Download Video File

Next, we'll download the sample colonoscopy video file that we'll use for the demo.

In [ ]:
!wget -q -O colon_video.mp4 "https://drive.google.com/uc?id=1QdpiIqWeDO0zBRVnPaqwBNtWxLyLbzZK&export=download"

## 3. Define the Processing Functions

Here is the Python code that will be used to process the video. It contains three main functions:
* `find_lumen`: Identifies the darkest area in a frame, assuming it's the lumen.
* `draw_navigation_ui`: Draws the user interface on the frame.
* `process_video`: The main function that reads the video, calls the other two functions for each frame, and yields the result.

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

def find_lumen(image):
    """
    Finds the lumen in a given video frame.
    
    This function currently uses a simple color thresholding method. It will likely need to be
    tuned or replaced with a more robust method for real-world videos.
    """
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lower_bound = np.array([0, 0, 0])
    upper_bound = np.array([180, 255, 100])
    mask = cv2.inRange(hsv, lower_bound, upper_bound)
    
    kernel = np.ones((15, 15), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return None, None
        
    largest_contour = max(contours, key=cv2.contourArea)
    
    M = cv2.moments(largest_contour)
    if M["m00"] == 0:
        return None, None
        
    center_x = int(M["m10"] / M["m00"])
    center_y = int(M["m01"] / M["m00"])
    
    return (center_x, center_y), largest_contour

def draw_navigation_ui(image, lumen_center, contour, frame_num):
    """Draws the navigation UI elements on the image."""
    height, width, _ = image.shape
    cv2.circle(image, (width // 2, height // 2), min(width, height) // 2 - 5, (0, 0, 0), 20)
    
    if contour is not None:
        cv2.drawContours(image, [contour], -1, (0, 255, 0), 2)
    if lumen_center is not None:
        cv2.circle(image, lumen_center, 7, (0, 0, 255), -1)
        img_center_x, img_center_y = width // 2, height // 2
        cv2.arrowedLine(image, (img_center_x, img_center_y), lumen_center, (255, 255, 0), 4)
    
    cv2.putText(image, "Lumen Navigation Demo", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    target_text = f"Target Lock: {lumen_center}" if lumen_center else "Target: Not Found"
    cv2.putText(image, target_text, (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(image, f"Frame: {frame_num}", (width - 160, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    return image

def process_video(video_path, max_frames=None):
    """Processes a video and yields each processed frame."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file at {video_path}")
        return

    frame_num = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or (max_frames is not None and frame_num >= max_frames):
            break
            
        detected_center, detected_contour = find_lumen(frame)
        output_image = draw_navigation_ui(frame.copy(), detected_center, detected_contour, frame_num)
        yield output_image
        
        frame_num += 1
            
    cap.release()
    print(f"\nSuccessfully processed all {frame_num} frames from the video.")

## 4. Run the Demo

Now, we'll run the `process_video` function on our downloaded video. We'll process a maximum of 100 frames to keep the demo quick. Each processed frame will be displayed below.

In [ ]:
for frame in process_video('colon_video.mp4', max_frames=100):
    cv2_imshow(frame)